# Exploratory Data Analysis on Numeric Columns and Features

The goal is to find a solution to preprocess data cleanly and then find high correlation and low p value features

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import re
from datetime import datetime

# ======================================================================
#          TEMPORALLY-AWARE GOING/TRACK CONDITION FEATURES
# ======================================================================

class TemporalGoingFeatureEngineer:
    """
    Creates track condition (going) performance features WITHOUT data leakage.
    Only uses historical data available BEFORE each race.
    """
    
    GOING_SCALE = [
        "Firm",
        "Good to Firm",
        "Good",
        "Good to Yielding",
        "Yielding",
        "Yielding to Soft",
        "Soft",
        "Soft to Heavy",
        "Heavy",
        "Standard"
    ]
    
    def __init__(self, df):
        self.df = df.copy()
        self._prepare_data()
    
    def _prepare_data(self):
        """Prepare track and going columns"""
        # Clean track name
        self.df['track_stripped'] = (
            self.df['track_name']
            .str.split('Racing Results')
            .str[0]
            .str.strip()
        )
        
        # Clean going (remove parentheses content)
        self.df['going_raw'] = (
            self.df['going']
            .str.split('(')
            .str[0]
            .str.strip()
        )
        
        # Extract standardized going
        self.df['going'] = self.df['going_raw'].apply(self._extract_clean_going)
    
    def _extract_clean_going(self, raw_going):
        """Extract standardized going from raw text"""
        if not isinstance(raw_going, str):
            return None
        
        raw = raw_going.lower().replace('-', ' ').strip()
        
        # Check for multi-word exact matches (longest first)
        for going in sorted(self.GOING_SCALE, key=lambda g: -len(g)):
            pattern = r'\b' + re.escape(going.lower()) + r'\b'
            if re.search(pattern, raw):
                return going
        
        # Check individual words
        tokens = re.findall(r'\b\w+\b', raw)
        for token in tokens:
            for going in self.GOING_SCALE:
                if token == going.lower():
                    return going
        
        return None
    
    def ensure_label(self):
        """Create binary winner label"""
        if 'label' not in self.df.columns:
            self.df['label'] = (self.df['race_position_clean'] == 1).astype(int)
        return self
    
    def ensure_race_date(self):
        """Ensure race_date column exists and is datetime"""
        if 'race_date' not in self.df.columns:
            # Extract from track_name if needed
            def parse_date(track_name):
                if not isinstance(track_name, str):
                    return None
                parts = track_name.split('|', 1)
                if len(parts) < 2:
                    return None
                date_str = parts[-1]
                # Remove ordinals
                cleaned = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str)
                try:
                    date_obj = datetime.strptime(cleaned.strip(), "%d %B %Y %H:%M")
                    return date_obj
                except:
                    return None
            
            self.df['race_date'] = self.df['track_name'].apply(parse_date)
        
        # Ensure datetime type
        self.df['race_date'] = pd.to_datetime(self.df['race_date'], errors='coerce')
        return self
    
    def add_temporal_performance_on_going(self):
        """
        Add performance_on_{going_type} features using ONLY historical data.
        
        CRITICAL: For each race, only uses races that occurred BEFORE that date.
        OPTIMIZED: Uses vectorized operations instead of nested loops.
        """
        self.ensure_label()
        self.ensure_race_date()
        
        # Sort by horse and date for cumulative calculations
        self.df = self.df.sort_values(['horse_name_clean', 'race_date']).reset_index(drop=True)
        
        # Get unique going types
        going_types = self.df['going'].dropna().unique()
        
        print(f"Creating temporal features for {len(going_types)} going types...")
        
        for going_type in going_types:
            feature_name = f'performance_on_{going_type}'
            
            # Create mask for this going type
            going_mask = self.df['going'] == going_type
            
            # Create binary indicator: 1 if race was on this going AND horse won
            self.df['_temp_win_on_going'] = (going_mask & (self.df['label'] == 1)).astype(int)
            
            # Create binary indicator: 1 if race was on this going
            self.df['_temp_race_on_going'] = going_mask.astype(int)
            
            # Calculate cumulative wins and races on this going (per horse)
            grouped = self.df.groupby('horse_name_clean')
            
            # Cumulative sum gives us total wins/races UP TO AND INCLUDING current race
            cumsum_wins = grouped['_temp_win_on_going'].cumsum()
            cumsum_races = grouped['_temp_race_on_going'].cumsum()
            
            # Shift to get stats BEFORE current race (not including it)
            wins_before = cumsum_wins - self.df['_temp_win_on_going']
            races_before = cumsum_races - self.df['_temp_race_on_going']
            
            # Calculate win rate (avoid division by zero)
            self.df[feature_name] = np.where(
                races_before > 0,
                wins_before / races_before,
                0.0
            )
            
            # Clean up temp columns
            self.df.drop(['_temp_win_on_going', '_temp_race_on_going'], axis=1, inplace=True)
            
            print(f"  ✓ {feature_name}")
        
        return self
    
    def add_going_numeric_encoding(self):
        """
        Add numeric encoding for going condition quality.
        Lower number = firmer ground, higher = softer ground.
        """
        going_to_num = {going: idx for idx, going in enumerate(self.GOING_SCALE)}
        self.df['condition_quality'] = self.df['going'].map(going_to_num)
        return self
    
    def calculate_feature_correlation(self, feature_col: str, target_col: str = 'label'):
        """
        Calculate Pearson correlation coefficient and p-value for a feature.
        
        Args:
            feature_col (str): Feature column name
            target_col (str): Target column name
            
        Returns:
            tuple: (correlation, p_value)
        """
        temp_df = self.df.dropna(subset=[feature_col, target_col]).copy()
        
        # Check for sufficient data and variation
        if len(temp_df) < 2 or temp_df[feature_col].nunique() == 1 or temp_df[target_col].nunique() == 1:
            return np.nan, np.nan
        
        correlation, p_value = pearsonr(
            temp_df[feature_col],
            temp_df[target_col]
        )
        
        return correlation, p_value
    
    def analyze_going_features(self, target_col='label'):
        """
        Analyze correlation for all going performance features.
        
        Args:
            target_col (str): Target column name
            
        Returns:
            pd.DataFrame: Sorted correlation results
        """
        # Get all performance_on_ features
        going_features = [col for col in self.df.columns 
                         if col.startswith('performance_on_')]
        
        # Add condition_quality if it exists
        if 'condition_quality' in self.df.columns:
            going_features.append('condition_quality')
        
        results = []
        
        print("\n" + "="*70)
        print("STATISTICAL FEATURE RANKING - GOING/TRACK CONDITION")
        print("="*70)
        print("\nAnalyzing correlations with target variable...")
        print()
        
        for feature in going_features:
            corr, p_val = self.calculate_feature_correlation(feature, target_col)
            
            # Determine significance
            if pd.isna(corr):
                status = "❌ Insufficient Data"
            elif p_val < 0.001:
                status = "✓✓✓ Highly Significant"
            elif p_val < 0.01:
                status = "✓✓ Very Significant"
            elif p_val < 0.05:
                status = "✓ Significant"
            else:
                status = "Not Significant"
            
            results.append({
                'Feature': feature,
                'Pearson_r': corr,
                'P_Value': p_val,
                'Status': status
            })
            
            # Print progress
            if not pd.isna(corr):
                print(f"✓ {feature:40s} | r={corr:+.6f} | p={p_val:.6f} | {status}")
            else:
                print(f"⚠ {feature:40s} | {status}")
        
        # Create results dataframe
        results_df = pd.DataFrame(results)
        
        # Sort by absolute correlation (strongest first)
        results_df['abs_corr'] = results_df['Pearson_r'].abs()
        results_df = results_df.sort_values('abs_corr', ascending=False)
        results_df = results_df.drop('abs_corr', axis=1)
        
        return results_df
    
    def interpret_correlation(self, r):
        """Interpret correlation strength"""
        abs_r = abs(r)
        
        if abs_r >= 0.7:
            strength = "Strong"
        elif abs_r >= 0.4:
            strength = "Moderate"
        elif abs_r >= 0.2:
            strength = "Weak"
        else:
            strength = "Very Weak"
        
        direction = "Positive" if r > 0 else "Negative"
        
        return f"{strength} {direction}"
    
    def create_correlation_report(self, target_col='label'):
        """
        Create comprehensive correlation report with interpretations.
        
        Args:
            target_col (str): Target column name
            
        Returns:
            pd.DataFrame: Detailed correlation report
        """
        results_df = self.analyze_going_features(target_col)
        
        # Add interpretation column
        results_df['Interpretation'] = results_df['Pearson_r'].apply(
            lambda x: self.interpret_correlation(x) if not pd.isna(x) else "N/A"
        )
        
        print("\n" + "="*70)
        print("GOING FEATURES CORRELATION REPORT")
        print("="*70)
        print("\nInterpretation Guide:")
        print("  r → +1.0 = Strong positive (higher feature → more likely to win)")
        print("  r → -1.0 = Strong negative (lower feature → more likely to win)")
        print("  p < 0.05 = Statistically significant relationship")
        print("\n")
        print(results_df.to_string(index=False))
        print("\n")
        
        return results_df
    
    def run_pipeline(self):
        """Execute full temporal feature engineering pipeline"""
        print("Starting temporal going feature engineering...")
        
        self.ensure_label()
        print("✓ Label created")
        
        self.ensure_race_date()
        print("✓ Race dates prepared")
        
        self.add_temporal_performance_on_going()
        print("✓ Temporal going performance features created")
        
        self.add_going_numeric_encoding()
        print("✓ Going numeric encoding added")
        
        print("\n✓ Pipeline complete!")
        return self.df


# ======================================================================
#                          USAGE EXAMPLE
# ======================================================================

if __name__ == "__main__":
    # Load data
    df = pd.read_csv('merged_horse_racing_data.csv')
    
    # Create engineer and run pipeline
    engineer = TemporalGoingFeatureEngineer(df)
    df_temporal = engineer.run_pipeline()
    
    # Create correlation report
    report = engineer.create_correlation_report(target_col='label')
    
    print("\n" + "="*70)
    print("TOP 10 GOING FEATURES BY CORRELATION STRENGTH")
    print("="*70)
    print(report.head(10).to_string(index=False))
    
    # # Save results
    # df_temporal.to_csv('horse_racing_temporal_going_features.csv', index=False)
    # report.to_csv('going_features_correlation_report.csv', index=False)
    
    print("\n✓ Dataset saved to 'horse_racing_temporal_going_features.csv'")
    print("✓ Report saved to 'going_features_correlation_report.csv'")

Starting temporal going feature engineering...
✓ Label created
✓ Race dates prepared
Creating temporal features for 9 going types...
  ✓ performance_on_Soft to Heavy
  ✓ performance_on_Good
  ✓ performance_on_Yielding
  ✓ performance_on_Soft
  ✓ performance_on_Yielding to Soft
  ✓ performance_on_Good to Yielding
  ✓ performance_on_Heavy
  ✓ performance_on_Standard
  ✓ performance_on_Good to Firm
✓ Temporal going performance features created
✓ Going numeric encoding added

✓ Pipeline complete!

STATISTICAL FEATURE RANKING - GOING/TRACK CONDITION

Analyzing correlations with target variable...

✓ performance_on_Soft to Heavy             | r=+0.024865 | p=0.000000 | ✓✓✓ Highly Significant
✓ performance_on_Good                      | r=+0.062745 | p=0.000000 | ✓✓✓ Highly Significant
✓ performance_on_Yielding                  | r=+0.031264 | p=0.000000 | ✓✓✓ Highly Significant
✓ performance_on_Soft                      | r=+0.040252 | p=0.000000 | ✓✓✓ Highly Significant
✓ performance_on_Yie

In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# ======================================================================
#           GOING FEATURES PREPROCESSING PIPELINE
# ======================================================================

class GoingFeaturesPreprocessor:
    """
    Temporal going features pipeline for horse racing prediction.
    Creates performance, binary, and interaction features WITHOUT data leakage.
    """
    
    GOING_SCALE = [
        "Firm", "Good to Firm", "Good", "Good to Yielding",
        "Yielding", "Yielding to Soft", "Soft", "Soft to Heavy",
        "Heavy", "Standard"
    ]
    
    # Top 10 features to keep (r >= 0.028)
    FEATURES_TO_KEEP = [
        'performance_on_Good',
        'active_perf_on_Good',
        'has_exp_on_Good',
        'performance_on_Good to Yielding',
        'performance_on_Soft',
        'has_exp_on_Good to Yielding',
        'has_exp_on_Soft',
        'performance_on_Yielding',
        'active_perf_on_Standard',
        'has_exp_on_Yielding'
    ]
    
    def __init__(self, verbose=True):
        self.verbose = verbose
    
    def _log(self, message):
        if self.verbose:
            print(message)
    
    def _extract_clean_going(self, raw_going):
        """Extract standardized going from raw text"""
        if not isinstance(raw_going, str):
            return None
        
        raw = raw_going.lower().replace('-', ' ').strip()
        
        # Check multi-word matches first (longest first)
        for going in sorted(self.GOING_SCALE, key=lambda g: -len(g)):
            pattern = r'\b' + re.escape(going.lower()) + r'\b'
            if re.search(pattern, raw):
                return going
        
        # Check individual words
        tokens = re.findall(r'\b\w+\b', raw)
        for token in tokens:
            for going in self.GOING_SCALE:
                if token == going.lower():
                    return going
        return None
    
    def _extract_race_date(self, track_name):
        """Extract race date from track_name column"""
        if not isinstance(track_name, str):
            return None
        
        new = track_name.split('|', 1)[-1]
        cleaned_string = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', new)
        
        try:
            date_object = datetime.strptime(cleaned_string, " %d %B %Y %H:%M")
            return date_object.strftime("%d/%m/%Y")
        except:
            return None
    
    def fit_transform(self, df):
        """
        Complete preprocessing pipeline.
        
        Args:
            df: Raw dataframe with columns: track_name, going, race_position_clean, horse_name_clean
        
        Returns:
            Processed dataframe with top 10 going features
        """
        self._log("="*70)
        self._log("GOING FEATURES PREPROCESSING PIPELINE")
        self._log("="*70)
        
        df = df.copy()
        
        # Step 1: Create label
        self._log("\n[1/7] Creating target label...")
        df['label'] = (df['race_position_clean'] == 1).astype(int)
        self._log(f"      Winners: {df['label'].sum():,} / {len(df):,} races")
        
        # Step 2: Extract and parse race date
        self._log("\n[2/7] Extracting race dates...")
        df['race_date'] = df['track_name'].apply(self._extract_race_date)
        df['race_date'] = pd.to_datetime(df['race_date'], format='%d/%m/%Y', errors='coerce')
        valid_dates = df['race_date'].notna().sum()
        self._log(f"      Valid dates: {valid_dates:,} / {len(df):,} ({valid_dates/len(df)*100:.1f}%)")
        
        # Step 3: Clean going column
        self._log("\n[3/7] Cleaning going conditions...")
        df['going_raw'] = df['going'].str.split('(').str[0].str.strip()
        df['going'] = df['going_raw'].apply(self._extract_clean_going)
        unique_goings = df['going'].nunique()
        self._log(f"      Unique going types: {unique_goings}")
        
        # Step 4: Sort for temporal calculation
        self._log("\n[4/7] Sorting by horse and date...")
        df = df.sort_values(['horse_name_clean', 'race_date']).reset_index(drop=True)
        
        # Step 5: Create temporal performance features
        self._log("\n[5/7] Creating temporal performance features...")
        going_types = df['going'].dropna().unique()
        
        for going_type in going_types:
            going_mask = df['going'] == going_type
            
            # Temporary columns for cumulative calculation
            df['_temp_win'] = (going_mask & (df['label'] == 1)).astype(int)
            df['_temp_race'] = going_mask.astype(int)
            
            # Cumulative sums per horse
            grouped = df.groupby('horse_name_clean')
            cumsum_wins = grouped['_temp_win'].cumsum()
            cumsum_races = grouped['_temp_race'].cumsum()
            
            # Shift to get BEFORE current race
            wins_before = cumsum_wins - df['_temp_win']
            races_before = cumsum_races - df['_temp_race']
            
            # Calculate win rate
            perf_col = f'performance_on_{going_type}'
            df[perf_col] = np.where(races_before > 0, wins_before / races_before, 0.0)
            
            df.drop(['_temp_win', '_temp_race'], axis=1, inplace=True)
        
        self._log(f"      Created {len(going_types)} performance features")
        
        # Step 6: Create binary and interaction features
        self._log("\n[6/7] Creating binary and interaction features...")
        
        perf_features = [col for col in df.columns if col.startswith('performance_on_')]
        
        for perf_feat in perf_features:
            going_type = perf_feat.replace('performance_on_', '')
            
            # Binary: has experience
            binary_feat = f'has_exp_on_{going_type}'
            df[binary_feat] = (df[perf_feat] > 0).astype(int)
            
            # Interaction: performance when racing on this going
            interaction_feat = f'active_perf_on_{going_type}'
            is_current_going = (df['going'] == going_type).astype(int)
            df[interaction_feat] = df[perf_feat] * is_current_going
        
        total_created = len(perf_features) * 3  # perf + binary + interaction
        self._log(f"      Created {total_created} total going features")
        
        # Step 7: Filter to top 10 features only
        self._log("\n[7/7] Filtering to top 10 features...")
        
        # Essential columns
        essential_cols = ['horse_name_clean', 'race_date', 'label', 'going']
        essential_cols = [c for c in essential_cols if c in df.columns]
        
        # Keep only top features that exist
        keep_features = [f for f in self.FEATURES_TO_KEEP if f in df.columns]
        
        final_cols = essential_cols + keep_features
        df_final = df[final_cols].copy()
        
        self._log(f"      Kept {len(keep_features)} / {total_created} features")
        self._log(f"      Final columns: {len(df_final.columns)} total")
        
        # Summary statistics
        self._log("\n" + "="*70)
        self._log("PIPELINE SUMMARY")
        self._log("="*70)
        self._log(f"Input rows:       {len(df):,}")
        self._log(f"Output rows:      {len(df_final):,}")
        self._log(f"Output columns:   {len(df_final.columns)}")
        self._log(f"Essential cols:   {len(essential_cols)}")
        self._log(f"Feature cols:     {len(keep_features)}")
        self._log(f"Memory usage:     {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        return df_final
    
    def get_feature_info(self):
        """Return information about the features being created"""
        info = {
            'total_features': len(self.FEATURES_TO_KEEP),
            'feature_types': {
                'performance': sum(1 for f in self.FEATURES_TO_KEEP if f.startswith('performance_')),
                'binary': sum(1 for f in self.FEATURES_TO_KEEP if f.startswith('has_exp_')),
                'interaction': sum(1 for f in self.FEATURES_TO_KEEP if f.startswith('active_'))
            },
            'features': self.FEATURES_TO_KEEP
        }
        return info


# ======================================================================
#                    JUPYTER CELL USAGE
# ======================================================================

# Load data
df = pd.read_csv('merged_horse_racing_data.csv')

# Initialize and run pipeline
preprocessor = GoingFeaturesPreprocessor(verbose=True)
df_processed = preprocessor.fit_transform(df)

# Display feature info
print("\n" + "="*70)
print("FEATURE BREAKDOWN")
print("="*70)
feature_info = preprocessor.get_feature_info()
print(f"Total features: {feature_info['total_features']}")
print(f"\nBy type:")
for feat_type, count in feature_info['feature_types'].items():
    print(f"  - {feat_type.capitalize()}: {count}")

print(f"\nFeature list:")
for i, feat in enumerate(feature_info['features'], 1):
    feat_type = feat.split('_')[0]
    print(f"  {i:2d}. {feat:40s} [{feat_type}]")

# Preview processed data
print("\n" + "="*70)
print("PROCESSED DATA PREVIEW")
print("="*70)
print(df_processed.head(10))

# Check for missing values
print("\n" + "="*70)
print("MISSING VALUES CHECK")
print("="*70)
missing = df_processed.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("✓ No missing values!")

# Save processed data (optional)
# df_processed.to_csv('processed_going_features.csv', index=False)
# print("\n✓ Saved to 'processed_going_features.csv'")

GOING FEATURES PREPROCESSING PIPELINE

[1/7] Creating target label...
      Winners: 8,780 / 106,250 races

[2/7] Extracting race dates...
      Valid dates: 80,094 / 106,250 (75.4%)

[3/7] Cleaning going conditions...
      Unique going types: 9

[4/7] Sorting by horse and date...

[5/7] Creating temporal performance features...
      Created 9 performance features

[6/7] Creating binary and interaction features...
      Created 27 total going features

[7/7] Filtering to top 10 features...
      Kept 10 / 27 features
      Final columns: 14 total

PIPELINE SUMMARY
Input rows:       106,250
Output rows:      106,250
Output columns:   14
Essential cols:   4
Feature cols:     10
Memory usage:     22.16 MB

FEATURE BREAKDOWN
Total features: 10

By type:
  - Performance: 4
  - Binary: 4
  - Interaction: 2

Feature list:
   1. performance_on_Good                      [performance]
   2. active_perf_on_Good                      [active]
   3. has_exp_on_Good                          [has]
 

In [3]:
import re
import pandas as pd
import numpy as np

def normalize_distance(df):
    
    def apply_patterns(text):
        # Handle non-string values
        if not isinstance(text, str):
            return np.nan
        
        text = text.strip()
        total_yards = 0
        
        # Pattern for compound distances like "1m 4f" or "1m4f"
        # Extract miles
        miles_match = re.search(r'(\d+\.?\d*)\s*[mM](?:iles?)?', text)
        if miles_match:
            miles = float(miles_match.group(1))
            total_yards += miles * 1760
        
        # Extract furlongs
        furlongs_match = re.search(r'(\d+\.?\d*)\s*[fF](?:urlongs?)?', text)
        if furlongs_match:
            furlongs = float(furlongs_match.group(1))
            total_yards += furlongs * 220
        
        # Extract yards
        yards_match = re.search(r'(\d+\.?\d*)\s*[yY](?:ards?)?', text)
        if yards_match:
            yards = float(yards_match.group(1))
            total_yards += yards
        
        # If we found any valid unit, return the total
        if total_yards > 0:
            return total_yards
        
        # Try to parse as plain number (assume yards)
        try:
            return float(text)
        except (ValueError, TypeError):
            return np.nan
    
    # Apply to your dataframe column
    df['distance_normalized'] = df['distance'].apply(apply_patterns)
    return df

# Load and process
df = pd.read_csv('merged_horse_racing_data.csv')
print("Original distance column sample:")
print(df['distance'].head(20))

df = normalize_distance(df)

# Print information about NaN values
print("\n" + "="*60)
print("ANALYZING NaN VALUES")
print("="*60)

nan_mask = df['distance_normalized'].isna()
nan_count = nan_mask.sum()
total_count = len(df)

print(f"\nTotal rows: {total_count}")
print(f"NaN values: {nan_count} ({nan_count/total_count*100:.2f}%)")
print(f"Valid values: {total_count - nan_count} ({(total_count-nan_count)/total_count*100:.2f}%)")

if nan_count > 0:
    print("\nOriginal distance values that resulted in NaN:")
    print("-" * 60)
    nan_originals = df.loc[nan_mask, 'distance']
    print(nan_originals.value_counts().head(30))
    
    print("\nFirst 20 examples of values that became NaN:")
    print(df.loc[nan_mask, ['distance']].head(20))

print("\n" + "="*60)
print("NORMALIZED VALUES SAMPLE")
print("="*60)
print(df[['distance', 'distance_normalized']].head(30))

# Plotting
import matplotlib.pyplot as plt
print(len(df))
df = df['distance_normalized'].dropna()
print(len(df))

bin_width = 200
bins = np.arange(column.min(), column.max() + bin_width, bin_width)

plt.figure(figsize=(10, 6))
plt.hist(column, bins=bins, edgecolor='black')
plt.xlabel('Distance (yards)')
plt.ylabel('Frequency')
plt.title(f'Distance Distribution (n={len(column):,}, dropped {nan_count:,} NaN values)')
plt.grid(axis='y', alpha=0.3)
plt.show()

# Print statistics
print("\nDistance statistics (yards):")
print(column.describe())

Original distance column sample:
0        7f
1        7f
2        7f
3        7f
4        7f
5        7f
6        7f
7        7f
8        7f
9        6f
10       6f
11       6f
12       6f
13       6f
14    1m 4f
15    1m 4f
16    1m 4f
17    1m 4f
18    1m 4f
19    1m 4f
Name: distance, dtype: object

ANALYZING NaN VALUES

Total rows: 106250
NaN values: 7 (0.01%)
Valid values: 106243 (99.99%)

Original distance values that resulted in NaN:
------------------------------------------------------------
Series([], Name: count, dtype: int64)

First 20 examples of values that became NaN:
      distance
42705      NaN
42706      NaN
42707      NaN
42708      NaN
42709      NaN
42710      NaN
42711      NaN

NORMALIZED VALUES SAMPLE
   distance  distance_normalized
0        7f               1540.0
1        7f               1540.0
2        7f               1540.0
3        7f               1540.0
4        7f               1540.0
5        7f               1540.0
6        7f               1540.0


NameError: name 'column' is not defined

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from datetime import datetime
import re
import matplotlib.pyplot as plt # Make sure matplotlib is imported

# --- _extract_race_date remains the same ---
def _extract_race_date(track_name):
    """Extract race date from track_name column"""
    if not isinstance(track_name, str):
        return None

    new = track_name.split('|', 1)[-1]
    cleaned_string = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', new)

    try:
        date_object = datetime.strptime(cleaned_string, " %d %B %Y %H:%M")
        return date_object
    except ValueError:
        try:
            date_object = datetime.strptime(cleaned_string, " %d %B %Y %H")
            return date_object
        except ValueError:
            return None

# --- normalize_distance function (assuming it exists from your original code) ---
# Placeholder - replace with your actual function if different
def normalize_distance(df):
    print("Assuming normalize_distance function exists and creates 'distance_normalized'.")
    # Example placeholder logic if 'distance' is in yards:
    if 'distance' in df.columns and 'distance_normalized' not in df.columns:
         # Basic check if it looks like furlongs/miles string
        if df['distance'].dtype == 'object':
             print("Warning: 'distance' is object type, attempting basic conversion.")
             # Very basic example, replace with your robust logic
             def rough_dist_conv(d):
                 try:
                     if 'm' in d and 'f' in d:
                         m, f = map(int, re.findall(r'(\d+)m|(\d+)f', d)[0])
                         return m * 1760 + f * 220
                     elif 'f' in d:
                         f = int(re.findall(r'(\d+)f', d)[0])
                         return f * 220
                     elif 'm' in d:
                          m = int(re.findall(r'(\d+)m', d)[0])
                          return m * 1760
                     return np.nan # Or handle other cases
                 except:
                     return np.nan
             df['distance_normalized'] = df['distance'].apply(rough_dist_conv)
        elif pd.api.types.is_numeric_dtype(df['distance']):
             print("Assuming 'distance' is already numeric (yards). Using as 'distance_normalized'.")
             df['distance_normalized'] = df['distance'] # Or apply your specific normalization
        else:
             print("Error: Cannot normalize distance. Unknown format.")
             # Create dummy column to avoid errors later, but this needs fixing
             df['distance_normalized'] = np.nan
    elif 'distance_normalized' not in df.columns:
        print("Error: 'distance' or 'distance_normalized' column not found.")
        df['distance_normalized'] = np.nan # Create dummy column
    return df


# --- MODIFIED: calculate_preferred_distance_temporal_fast ---
def calculate_preferred_distance_temporal_fast(df):
    """
    Calculate different versions of horse's preferred distance based ONLY on past races.
    - preferred_distance_top3_only: Avg distance of past top 3 finishes (NaN if none).
    - avg_past_distance_all: Avg distance of ALL valid past races (NaN if none).
    - preferred_distance_fallback: Uses top3, falls back to all if no top3 yet.
    """
    # Sort by horse and date
    df = df.sort_values(['horse_name_clean', 'race_date']).reset_index(drop=True)

    print("Sample of data (sorted by date):")
    print(df[['horse_name_clean', 'race_date', 'race_position_clean', 'distance_normalized']].head(10))
    print(f"\nTotal rows before calculations: {len(df)}")

    # Create helper columns
    df['valid_race'] = (df['race_position_clean'] >= 1) & (df['distance_normalized'].notna())
    df['top_3_finish'] = (df['race_position_clean'] <= 3) & df['valid_race']
    df['distance_x_top3'] = df['distance_normalized'].where(df['top_3_finish'], 0) # Use where for clarity
    df['distance_x_valid'] = df['distance_normalized'].where(df['valid_race'], 0) # Use where for clarity

    print("Calculating expanding statistics...")
    grouped = df.groupby('horse_name_clean', group_keys=False) # group_keys=False is slightly faster

    # --- Calculate cumulative counts and sums (using transform for efficiency) ---
    # Shift(1) ensures we only use PAST races
    df['cumsum_top3'] = grouped['top_3_finish'].transform(lambda x: x.shift(1, fill_value=0).cumsum())
    df['cumsum_distance_top3'] = grouped['distance_x_top3'].transform(lambda x: x.shift(1, fill_value=0).cumsum())
    df['cumsum_races_run'] = grouped['valid_race'].transform(lambda x: x.shift(1, fill_value=0).cumsum())
    df['cumsum_distance_all'] = grouped['distance_x_valid'].transform(lambda x: x.shift(1, fill_value=0).cumsum())

    # --- Calculate the different preferred distance metrics ---

    # 1. Original: Average distance of past top 3 finishes (NaN if none)
    temp_denom_top3 = df['cumsum_top3'].replace(0, np.nan)
    df['preferred_distance_top3_only'] = df['cumsum_distance_top3'] / temp_denom_top3

    # 2. Alternative A: Average distance of ALL valid past races (NaN if none)
    temp_denom_all = df['cumsum_races_run'].replace(0, np.nan)
    df['avg_past_distance_all'] = df['cumsum_distance_all'] / temp_denom_all

    # 3. Alternative B (Fallback): Use top3 average, but use 'all race average' if no prior top 3
    # We can use fillna here because preferred_distance_top3_only is already NaN where cumsum_top3 is 0
    df['preferred_distance_fallback'] = df['preferred_distance_top3_only'].fillna(df['avg_past_distance_all'])

    print(f"Preferred distances (top3 only) calculated for {df['preferred_distance_top3_only'].notna().sum()} races")
    print(f"Average past distances (all) calculated for {df['avg_past_distance_all'].notna().sum()} races")
    print(f"Preferred distances (fallback) calculated for {df['preferred_distance_fallback'].notna().sum()} races")

    # Clean up temporary columns
    df = df.drop(columns=['valid_race', 'top_3_finish', 'distance_x_top3', 'distance_x_valid',
                          'cumsum_top3', 'cumsum_distance_top3', 'cumsum_races_run', 'cumsum_distance_all'])

    return df

# --- MODIFIED: calculate_distance_match_and_correlation ---
def calculate_distance_match_and_correlation(df, pref_dist_col):
    """
    Calculate if current distance matches a specific preferred distance column
    and compute correlation with winning.

    Args:
        df (pd.DataFrame): DataFrame containing results and the preferred distance column.
        pref_dist_col (str): The name of the preferred distance column to analyze.

    Returns:
        tuple: (DataFrame used for analysis, dict containing correlation stats)
    """
    print(f"\n--- Analyzing Preferred Distance Column: {pref_dist_col} ---")

    # Ensure winner column exists
    if 'winner' not in df.columns:
        df['winner'] = (df['race_position_clean'] == 1).astype(int)

    # Calculate absolute difference using the specified preferred distance column
    df[f'{pref_dist_col}_diff'] = np.abs(df[pref_dist_col] - df['distance_normalized'])

    # Create binary feature: is distance within tolerance of preferred?
    tolerance = 220  # yards
    df[f'{pref_dist_col}_match'] = (df[f'{pref_dist_col}_diff'] <= tolerance).astype(int)

    # --- Analysis Part ---
    # Remove rows where we don't have the specific preferred distance or winner
    df_analysis = df.dropna(subset=[pref_dist_col, 'winner', f'{pref_dist_col}_diff']).copy() # Use copy to avoid SettingWithCopyWarning in plotting

    if df_analysis.empty:
        print("No valid races for analysis after dropping NaNs.")
        return df_analysis, {} # Return empty results

    print("\n" + "="*60)
    print(f"DISTANCE MATCH ANALYSIS ({pref_dist_col})")
    print("="*60)
    print(f"Total races analyzed: {len(df_analysis)}")
    match_col = f'{pref_dist_col}_match'
    diff_col = f'{pref_dist_col}_diff'
    print(f"Races with distance match: {df_analysis[match_col].sum()}")
    print(f"Races without distance match: {len(df_analysis) - df_analysis[match_col].sum()}")

    # Calculate win rates
    overall_win_rate = df_analysis['winner'].mean()
    # Handle cases where one category might be empty (though unlikely with NaN drop)
    win_rate_with_match = df_analysis[df_analysis[match_col] == 1]['winner'].mean() if (df_analysis[match_col] == 1).any() else 0
    win_rate_without_match = df_analysis[df_analysis[match_col] == 0]['winner'].mean() if (df_analysis[match_col] == 0).any() else 0

    print(f"\nOverall win rate: {overall_win_rate:.4f} ({overall_win_rate*100:.2f}%)")
    print(f"Win rate when distance matches: {win_rate_with_match:.4f} ({win_rate_with_match*100:.2f}%)")
    print(f"Win rate when distance doesn't match: {win_rate_without_match:.4f} ({win_rate_without_match*100:.2f}%)")
    print(f"Lift: {(win_rate_with_match/overall_win_rate) if overall_win_rate > 0 else 'N/A'}") # Avoid division by zero for lift

    results = {
        'overall_win_rate': overall_win_rate,
        'win_rate_with_match': win_rate_with_match,
        'win_rate_without_match': win_rate_without_match
    }

    # Pearson correlation between distance_match and winner
    print("\n" + "="*60)
    print(f"PEARSON CORRELATION: {match_col.upper()} vs WINNER")
    print("="*60)

    if df_analysis[match_col].nunique() > 1 and df_analysis['winner'].nunique() > 1: # Check for variance
        r_match, p_match = pearsonr(df_analysis[match_col], df_analysis['winner'])
        print(f"Correlation (r): {r_match:.6f}")
        print(f"P-value: {p_match:.6e}")
        print(f"Significant at p<0.05: {'YES' if p_match < 0.05 else 'NO'}")
        print(f"Significant at p<0.01: {'YES' if p_match < 0.01 else 'NO'}")
        results[f'{pref_dist_col}_match_r'] = r_match
        results[f'{pref_dist_col}_match_p'] = p_match
    else:
        print("Skipping correlation: Insufficient variance in match or winner.")
        results[f'{pref_dist_col}_match_r'] = np.nan
        results[f'{pref_dist_col}_match_p'] = np.nan


    # Also calculate correlation with continuous distance difference
    print("\n" + "="*60)
    print(f"PEARSON CORRELATION: {diff_col.upper()} vs WINNER")
    print("="*60)

    if df_analysis[diff_col].nunique() > 1 and df_analysis['winner'].nunique() > 1: # Check for variance
        # Negative correlation expected (smaller diff = better)
        r_diff, p_diff = pearsonr(df_analysis[diff_col], df_analysis['winner'])
        print(f"Correlation (r): {r_diff:.6f}")
        print(f"P-value: {p_diff:.6e}")
        print(f"Significant at p<0.05: {'YES' if p_diff < 0.05 else 'NO'}")
        print(f"Significant at p<0.01: {'YES' if p_diff < 0.01 else 'NO'}")
        results[f'{pref_dist_col}_diff_r'] = r_diff
        results[f'{pref_dist_col}_diff_p'] = p_diff
    else:
         print("Skipping correlation: Insufficient variance in difference or winner.")
         results[f'{pref_dist_col}_diff_r'] = np.nan
         results[f'{pref_dist_col}_diff_p'] = np.nan


    return df_analysis, results


# === Main Execution ===

# Load data
print("Loading data...")
df = pd.read_csv('merged_horse_racing_data.csv')

# Extract and convert race_date to datetime
print("Extracting race dates...")
df['race_date'] = df['track_name'].apply(_extract_race_date)
df = df.dropna(subset=['race_date']) # Drop rows where date couldn't be parsed

print(f"Data loaded: {len(df)} rows")

# Normalize distance
print("Normalizing distances...")
df = normalize_distance(df) # Make sure this function exists and works
df = df.dropna(subset=['distance_normalized']) # Drop rows where distance couldn't be normalized
print(f"Rows after distance normalization & dropna: {len(df)}")


# Calculate preferred distance variants with temporal awareness
print("\nCalculating preferred distances using only past races (VECTORIZED)...")
df = calculate_preferred_distance_temporal_fast(df)

# --- Analyze correlation for EACH preferred distance variant ---
all_correlation_stats = {}

# Analyze original (top 3 only, might have NaNs)
df_analysis_top3, stats_top3 = calculate_distance_match_and_correlation(df, 'preferred_distance_top3_only')
all_correlation_stats['top3_only'] = stats_top3

# Analyze alternative A (all past races)
df_analysis_all, stats_all = calculate_distance_match_and_correlation(df, 'avg_past_distance_all')
all_correlation_stats['all_past'] = stats_all

# Analyze alternative B (fallback)
df_analysis_fallback, stats_fallback = calculate_distance_match_and_correlation(df, 'preferred_distance_fallback')
all_correlation_stats['fallback'] = stats_fallback


# --- Display Combined Correlation Results ---
print("\n\n" + "="*70)
print("COMBINED CORRELATION SUMMARY")
print("="*70)
corr_summary = []
for key, stats in all_correlation_stats.items():
    if stats: # Check if stats dictionary is not empty
        corr_summary.append({
            'Method': key,
            'Match_Corr (r)': stats.get(f'{key}_match_r', np.nan),
            'Match_PVal': stats.get(f'{key}_match_p', np.nan),
            'Diff_Corr (r)': stats.get(f'{key}_diff_r', np.nan),
            'Diff_PVal': stats.get(f'{key}_diff_p', np.nan),
            'WinRate_Match': stats.get('win_rate_with_match', np.nan),
            'WinRate_NoMatch': stats.get('win_rate_without_match', np.nan)
        })

if corr_summary:
    summary_corr_df = pd.DataFrame(corr_summary)
    print(summary_corr_df.to_markdown(index=False, floatfmt=".4f"))
else:
    print("No correlation results to display.")


# --- Visualizations (Using the Fallback version for plotting example) ---
# You might want to plot for others too, or create comparison plots
if not df_analysis_fallback.empty:
    print("\n\n--- Generating Plots (Using Fallback Method for Example) ---")
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Distance Match Analysis (Fallback Preferred Distance)', fontsize=16)

    pref_dist_col = 'preferred_distance_fallback'
    match_col = f'{pref_dist_col}_match'
    diff_col = f'{pref_dist_col}_diff'

    # Plot 1: Win rate by distance match
    win_rates = df_analysis_fallback.groupby(match_col)['winner'].mean()
    axes[0, 0].bar(['No Match', 'Match'], win_rates.values, color=['coral', 'skyblue'], edgecolor='black')
    axes[0, 0].set_ylabel('Win Rate')
    axes[0, 0].set_title('Win Rate: Distance Match vs No Match')
    axes[0, 0].grid(axis='y', alpha=0.3)
    axes[0,0].set_ylim(bottom=0) # Start y-axis at 0
    for i, v in enumerate(win_rates.values):
        axes[0, 0].text(i, v + 0.002, f'{v:.4f}', ha='center', va='bottom')

    # Plot 2: Distribution of distance differences
    axes[0, 1].hist(df_analysis_fallback[diff_col].dropna(), bins=50, edgecolor='black', alpha=0.7) # Ensure dropna for hist
    axes[0, 1].axvline(220, color='red', linestyle='--', label='1 Furlong (220 yards)')
    axes[0, 1].set_xlabel('Distance Difference (yards)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Distance Differences')
    axes[0, 1].legend()
    axes[0, 1].grid(axis='y', alpha=0.3)

    # Plot 3: Win rate by distance difference bins
    df_analysis_fallback['distance_diff_bin'] = pd.cut(df_analysis_fallback[diff_col],
                                               bins=[0, 110, 220, 330, 440, 660, 880, np.inf], # Finer bins near zero
                                               labels=['0-110y','110-220y', '220-330y', '330-440y', '440-660y', '660-880y', '880y+'],
                                               right=False) # Include 0 in first bin
    # Use observed=False to include potentially empty bins if using Categorical
    win_by_diff = df_analysis_fallback.groupby('distance_diff_bin', observed=False)['winner'].mean()
    axes[1, 0].bar(range(len(win_by_diff)), win_by_diff.values, edgecolor='black')
    axes[1, 0].set_xticks(range(len(win_by_diff)))
    axes[1, 0].set_xticklabels(win_by_diff.index, rotation=45, ha='right')
    axes[1, 0].set_ylabel('Win Rate')
    axes[1, 0].set_xlabel('Distance Difference Range')
    axes[1, 0].set_title('Win Rate by Distance Difference')
    axes[1, 0].grid(axis='y', alpha=0.3)
    axes[1,0].set_ylim(bottom=0) # Start y-axis at 0


    # Plot 4: Sample counts by bin
    sample_counts = df_analysis_fallback.groupby('distance_diff_bin', observed=False).size()
    axes[1, 1].bar(range(len(sample_counts)), sample_counts.values, edgecolor='black', color='lightgreen')
    axes[1, 1].set_xticks(range(len(sample_counts)))
    axes[1, 1].set_xticklabels(sample_counts.index, rotation=45, ha='right')
    axes[1, 1].set_ylabel('Number of Races')
    axes[1, 1].set_xlabel('Distance Difference Range')
    axes[1, 1].set_title('Sample Size by Distance Difference')
    axes[1, 1].grid(axis='y', alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()

    # Summary table (Fallback)
    print("\n" + "="*60)
    print(f"SUMMARY STATISTICS ({pref_dist_col})")
    print("="*60)
    # Use observed=False for groupby if 'match_col' could become categorical
    summary_df = df_analysis_fallback.groupby(match_col, observed=False).agg(
        Total_Races=('winner', 'count'),
        Wins=('winner', 'sum'),
        Win_Rate=('winner', 'mean'),
        Avg_Distance_Diff=(diff_col, 'mean')
    ).round(4)
    summary_df.index = ['No Match', 'Match'] # Rename index for clarity
    print(summary_df)

else:
    print("\n\n--- Skipping Plots and Summary: No data available for fallback analysis ---")


# Save final df with all new columns
print("\nSaving results...")
df.to_csv('merged_horse_racing_data_with_preferred_distance_variants.csv', index=False)
print("Data saved to 'merged_horse_racing_data_with_preferred_distance_variants.csv'")

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, average_precision_score
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import scipy.stats as stats
from datetime import datetime
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ============================================================
#                    FEATURE ENGINEERING
# ============================================================

def clean_names(name):
    """Clean jockey/trainer names to standard format"""
    if not isinstance(name, str):
        return name
    
    name = name.replace('.', '').strip()
    parts = name.split()
    
    if len(parts) < 2:
        return name.title()
    
    initials = [p[0].upper() + '.' for p in parts[:-1]]
    last_name = parts[-1]
    
    return ' '.join(initials + [last_name])

def wilson_score(wins, total, confidence=0.95):
    """Calculate Wilson score for confidence interval"""
    if total == 0:
        return 0.0
    z = {0.95: 1.96, 0.99: 2.576, 0.90: 1.645}[confidence]
    phat = wins / total
    denominator = 1 + z**2 / total
    centre = phat + z**2 / (2 * total)
    margin = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * total)) / total)
    return (centre - margin) / denominator

def extract_clean_going(raw_going):
    """Extract standardized going condition"""
    GOING_SCALE = [
        "Firm", "Good to Firm", "Good", "Good to Yielding",
        "Yielding", "Yielding to Soft", "Soft", "Soft to Heavy",
        "Heavy", "Standard"
    ]
    
    if not isinstance(raw_going, str):
        return None

    raw = raw_going.lower().replace('-', ' ').strip()

    for going in sorted(GOING_SCALE, key=lambda g: -len(g)):
        pattern = r'\b' + re.escape(going.lower()) + r'\b'
        if re.search(pattern, raw):
            return going

    tokens = re.findall(r'\b\w+\b', raw)
    for token in tokens:
        for going in GOING_SCALE:
            if token == going.lower():
                return going

    return None

def normalize_distance(dist_str: str) -> float:
    """
    Convert a horse-race distance string (e.g. '2m 6f 74y') into total yards.
    Handles NaN and missing values.
    """
    dist_str = str(dist_str).strip()
    
    # Handle missing/invalid values
    if dist_str in ['nan', 'NaN', 'N/A', '-', '', 'None']:
        return np.nan
    
    pattern = (
    r'^\s*'
    r'(?:(?P<miles>\d+(?:\.\d+)?)\s*m)?\s*'
    r'(?:(?P<furlongs>\d+(?:\.\d+)?)\s*f)?\s*'
    r'(?:(?P<yards>\d+(?:\.\d+)?)\s*y)?'
    r'\s*$'
)
    
    m = re.match(pattern, dist_str, flags=re.IGNORECASE)
    if not m:
        print(f"Warning: Invalid distance format: '{dist_str}' - returning NaN")
        return np.nan
    
    miles = float(m.group('miles') or 0)
    furlongs = float(m.group('furlongs') or 0)
    yards = float(m.group('yards') or 0)
    return float(miles * 1760 + furlongs * 220 + yards)

def calculate_form_features(races):
    """Calculate EMA form features for a horse's race history"""
    if len(races) < 2:
        return {}
    
    races = races.sort_values('race_date_clean').reset_index(drop=True)
    
    form_scores = []
    for _, race in races.iterrows():
        if pd.isna(race['race_position_clean']) or pd.isna(race['rating_clean']):
            form_scores.append(np.nan)
            continue
            
        try:
            position = float(race['race_position_clean'])
            rating = float(race['rating_clean']) if not pd.isna(race['rating_clean']) else 70
            
            norm_pos = max(0, 1 - (position - 1) / 19)
            norm_rating = rating / 130
            
            form_score = 0.6 * norm_pos + 0.4 * norm_rating
            form_scores.append(form_score)
        except:
            form_scores.append(np.nan)
    
    features = {}
    if len(form_scores) >= 2:
        ema = form_scores[0] if not pd.isna(form_scores[0]) else 0.5
        for score in form_scores[1:]:
            if not pd.isna(score):
                ema = 0.3 * score + 0.7 * ema
        features['EMA_Form'] = ema
    
    return features

def normalize_weight(w_str: str) -> float:
    """
    Normalize British weight format to pounds.
    
    Handles formats like:
    - '10-13' → 10 stone 13 pounds → 153 pounds
    - '11-0tp' → 11 stone 0 pounds → 154 pounds (ignoring suffix)
    - '10-13tb1' → 10 stone 13 pounds → 153 pounds (ignoring suffix)
    - '-' or empty → returns np.nan
    """
    w_str = str(w_str).strip()
    
    if w_str in ['-', '', 'N/A', 'nan', 'NaN', 'None']:
        return np.nan
    
    # British stone-pounds format: "stone-pounds" optionally followed by letters
    stone_pounds_match = re.match(r'^(\d+)-(\d+)', w_str)
    if stone_pounds_match:
        stone = int(stone_pounds_match.group(1))
        pounds = int(stone_pounds_match.group(2))
        return stone * 14 + pounds  # 1 stone = 14 pounds
    
    # Fallback: extract leading number (assuming already in pounds)
    match = re.match(r'^(\d+(?:\.\d+)?)', w_str)
    if match:
        return float(match.group(1))
    
    print(f"Warning: Invalid weight format: '{w_str}' - returning NaN")
    return np.nan

def process_features(df):
    """Main feature processing pipeline"""
    print("Starting feature engineering...")
    
    # Normalize distance column if it exists
    if 'distance' in df.columns:
        print("Normalizing distance values...")
        df['distance'] = df['distance'].apply(normalize_distance)
        valid_distances = df['distance'].dropna()
        print(f"  Normalized {len(valid_distances)} valid distances out of {len(df)} total")
        if len(valid_distances) > 0:
            print(f"  Distance range: {valid_distances.min():.0f} - {valid_distances.max():.0f} yards")
    
    # Normalize weight column if it exists
    if 'weight' in df.columns:
        print("Normalizing weight values...")
        # Show example of weight conversion for debugging
        sample_weights = df['weight'].dropna().head(3)
        if len(sample_weights) > 0:
            print(f"  Sample weight conversions: {list(sample_weights.values)} → ", end="")
        
        df['weight'] = df['weight'].apply(normalize_weight)
        valid_weights = df['weight'].dropna()
        
        if len(sample_weights) > 0:
            print(f"{list(df['weight'].iloc[sample_weights.index].values)}")
        
        print(f"  Normalized {len(valid_weights)} valid weights out of {len(df)} total")
        if len(valid_weights) > 0:
            print(f"  Weight range: {valid_weights.min():.0f} - {valid_weights.max():.0f} pounds")
    
    # Ensure numeric columns are properly typed
    numeric_columns = ['age', 'claims']
    if 'rating_clean' in df.columns:
        numeric_columns.append('rating_clean')
    elif 'rating' in df.columns:
        numeric_columns.append('rating')
    
    for col in numeric_columns:
        if col in df.columns:
            # Convert to numeric, coercing errors to NaN
            df[col] = pd.to_numeric(df[col], errors='coerce')
            print(f"  Converted {col} to numeric: {df[col].notna().sum()} valid values")
    
    df['J_Name'] = df['jockey'].apply(clean_names).str.replace('.','', regex=False)
    df['T_Name'] = df['trainer'].apply(clean_names).str.replace('.','', regex=False)
    df['track_stripped'] = df['track_name'].str.split('|').str[0].str.strip()
    
    # Jockey-Trainer combinations
    df['j_t'] = df['J_Name'].astype(str) + " | " + df['T_Name'].astype(str)
    
    # Create label
    df['label'] = (df['race_position_clean'] == 1).astype(int)
    
    # Calculate JT wins and runs
    jt_wins = df[df['label'] == 1].groupby('j_t').size().reset_index(name='jt_wins')
    df = df.merge(jt_wins, on='j_t', how='left')
    df['jt_wins'] = df['jt_wins'].fillna(0).astype(int)
    
    jt_runs = df['j_t'].value_counts().reset_index()
    jt_runs.columns = ['j_t', 'jt_runs']
    df = df.merge(jt_runs, on='j_t', how='left')
    
    # Wilson score
    df['wilson_score'] = df.apply(
        lambda row: wilson_score(row['jt_wins'], row['jt_runs']),
        axis=1
    )
    
    # Clean going conditions
    df['going'] = df['going'].str.split('(').str[0].str.strip()
    df['going'] = df['going'].apply(extract_clean_going)
    
    # Performance on going
    for going_type in df['going'].dropna().unique():
        going_mask = df['going'] == going_type
        perf = df.loc[going_mask].groupby('horse_name')['label'].mean()
        df[f'performance_on_{going_type}'] = df['horse_name'].map(perf).fillna(0)
    
    # Calculate EMA Form
    df['EMA_Form'] = 0.5  # Default value
    for horse in df['horse_name'].unique():
        horse_races = df[df['horse_name'] == horse].copy()
        if len(horse_races) >= 2:
            form_features = calculate_form_features(horse_races)
            if 'EMA_Form' in form_features:
                df.loc[df['horse_name'] == horse, 'EMA_Form'] = form_features['EMA_Form']
    
    # Historical performance features
    df = df.sort_values('race_date_clean') if 'race_date_clean' in df.columns else df.sort_index()
    
    df['recent_win_rate'] = 0.0
    df['career_wins'] = 0
    df['career_races'] = 0
    df['career_win_rate'] = 0.0
    df['days_since_last'] = 0
    
    for horse in df['horse_name'].unique():
        horse_mask = df['horse_name'] == horse
        horse_indices = df[horse_mask].index.tolist()
        
        for i, idx in enumerate(horse_indices):
            if i > 0:
                prior_indices = horse_indices[:i]
                prior_labels = df.loc[prior_indices, 'label']
                
                career_wins = prior_labels.sum()
                career_races = len(prior_labels)
                career_win_rate = career_wins / career_races if career_races > 0 else 0
                
                recent_labels = prior_labels.tail(5) if len(prior_labels) >= 5 else prior_labels
                recent_win_rate = recent_labels.mean() if len(recent_labels) > 0 else 0
                
                df.loc[idx, 'recent_win_rate'] = recent_win_rate
                df.loc[idx, 'career_wins'] = career_wins
                df.loc[idx, 'career_races'] = career_races
                df.loc[idx, 'career_win_rate'] = career_win_rate
    
    # Track-specific features
    track_stats = []
    for idx in df.index:
        track = df.loc[idx, 'track_stripped']
        other_races = df[(df['track_stripped'] == track) & (df.index != idx)]
        
        if len(other_races) >= 10:
            track_win_rate = other_races['label'].mean()
            if 'distance' in df.columns:
                # Use nanmean to handle NaN values properly
                track_avg_distance = other_races['distance'].dropna().mean() if len(other_races['distance'].dropna()) > 0 else df['distance'].dropna().mean()
            else:
                track_avg_distance = 0
        else:
            track_win_rate = df['label'].mean()
            if 'distance' in df.columns:
                track_avg_distance = df['distance'].dropna().mean() if len(df['distance'].dropna()) > 0 else 0
            else:
                track_avg_distance = 0
        
        track_stats.append({
            'index': idx,
            'track_win_rate': track_win_rate,
            'track_avg_distance': track_avg_distance
        })
    
    stats_df = pd.DataFrame(track_stats).set_index('index')
    df['track_win_rate'] = stats_df['track_win_rate']
    df['track_avg_distance'] = stats_df['track_avg_distance']
    
    # Weight-related features
    if 'weight' in df.columns:
        # Only calculate percentile if we have valid weight values
        if df['weight'].notna().sum() > 0:
            df['weight_percentile'] = df['weight'].rank(pct=True)
        else:
            df['weight_percentile'] = 0.5  # Default to middle if no valid weights
    
    # Age-related features
    if 'age' in df.columns:
        df['is_young_horse'] = (df['age'] <= 3).astype(int)
        df['is_prime_age'] = ((df['age'] >= 4) & (df['age'] <= 6)).astype(int)
        df['is_veteran'] = (df['age'] >= 7).astype(int)
    
    print("Feature engineering complete!")
    return df

# ============================================================
#                    TABNET TRAINING
# ============================================================

def prepare_data_for_tabnet(df):
    """Prepare data for TabNet training"""
    
    # Define feature columns
    high_correlation_features = [
        'EMA_Form', 'wilson_score', 'jt_runs', 'jt_wins'
    ]
    
    # Add performance_on_{going_type} features
    going_features = [col for col in df.columns if col.startswith('performance_on_')]
    high_correlation_features.extend(going_features)
    
    additional_features = [
        'days_since_last', 'track_win_rate', 'track_avg_distance',
        'weight_percentile', 'is_young_horse', 'is_prime_age', 'is_veteran',
        'recent_win_rate', 'career_wins', 'career_races', 'career_win_rate'
    ]
    
    # Raw numeric features
    raw_features = ['distance', 'claims', 'age', 'weight']
    # Add rating_clean or rating if available
    if 'rating_clean' in df.columns:
        raw_features.append('rating_clean')
    elif 'rating' in df.columns:
        raw_features.append('rating')
    
    # Check which features exist in the dataframe
    all_features = []
    for feat in high_correlation_features + additional_features + raw_features:
        if feat in df.columns:
            all_features.append(feat)
    
    print(f"Total features for training: {len(all_features)}")
    print(f"High correlation features: {[f for f in high_correlation_features if f in df.columns]}")
    
    # Handle missing values
    for col in all_features:
        if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            # Use median for numeric columns, but handle empty columns
            if df[col].notna().sum() > 0:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(0)
    
    # Prepare feature matrix
    X = df[all_features].values
    y = df['label'].values
    
    # Create attention mask (1 for high correlation features, 0 for others)
    attention_mask = np.zeros(len(all_features))
    for i, feat in enumerate(all_features):
        if feat in high_correlation_features:
            attention_mask[i] = 1
    
    return X, y, all_features, attention_mask

def train_tabnet(X_train, y_train, X_val, y_val, attention_mask):
    """Train TabNet model"""
    
    # TabNet parameters
    tabnet_params = {
        'n_d': 64,                    # Width of decision prediction layer
        'n_a': 64,                    # Width of attention embedding
        'n_steps': 5,                 # Number of decision steps
        'gamma': 1.5,                 # Relaxation parameter
        'n_independent': 2,           # Number of independent GLU layers
        'n_shared': 2,                # Number of shared GLU layers
        'epsilon': 1e-15,
        'momentum': 0.02,
        'lambda_sparse': 1e-4,        # Sparsity loss weight
        'optimizer_fn': torch.optim.Adam,
        'optimizer_params': dict(lr=2e-2),
        'scheduler_fn': torch.optim.lr_scheduler.StepLR,
        'scheduler_params': {"step_size": 50, "gamma": 0.9},
        'mask_type': 'sparsemax',
        'verbose': 1,
        'device_name': 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed': 42
    }
    
    # Initialize TabNet
    clf = TabNetClassifier(**tabnet_params)
    
    # Train the model
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_name=['val'],
        eval_metric=['auc', 'accuracy'],
        max_epochs=200,
        patience=20,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
        augmentations=None  # No augmentation for tabular data
    )
    
    return clf

# ============================================================
#                    THRESHOLD TESTING & ANALYSIS
# ============================================================

def test_thresholds(y_true, y_proba, thresholds=None):
    """
    Test different probability thresholds and calculate metrics
    
    Args:
        y_true: True binary labels
        y_proba: Predicted probabilities
        thresholds: List of thresholds to test (default: 0.1 to 0.9 in steps of 0.05)
    
    Returns:
        DataFrame with threshold analysis results
    """
    if thresholds is None:
        thresholds = np.arange(0.1, 0.95, 0.05)
    
    results = []
    
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        
        # Calculate metrics
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        tn = np.sum((y_true == 0) & (y_pred == 0))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        # Positive predictive value and negative predictive value
        ppv = precision  # Same as precision
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        
        results.append({
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'accuracy': accuracy,
            'specificity': specificity,
            'ppv': ppv,
            'npv': npv,
            'tp': tp,
            'fp': fp,
            'tn': tn,
            'fn': fn,
            'predicted_positive': tp + fp,
            'predicted_negative': tn + fn
        })
    
    return pd.DataFrame(results)

def plot_precision_recall_curve(y_true, y_proba, save_path='precision_recall_curve.png'):
    """
    Plot precision-recall curve
    
    Args:
        y_true: True binary labels
        y_proba: Predicted probabilities
        save_path: Path to save the plot
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    avg_precision = average_precision_score(y_true, y_proba)
    
    plt.figure(figsize=(10, 8))
    
    # Main precision-recall curve
    plt.subplot(2, 1, 1)
    plt.plot(recall, precision, linewidth=2, label=f'PR Curve (AP = {avg_precision:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Precision and recall vs threshold
    plt.subplot(2, 1, 2)
    plt.plot(thresholds, precision[:-1], label='Precision', linewidth=2)
    plt.plot(thresholds, recall[:-1], label='Recall', linewidth=2)
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.title('Precision and Recall vs Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return avg_precision

def find_optimal_threshold(threshold_results, metric='f1_score'):
    """
    Find optimal threshold based on specified metric
    
    Args:
        threshold_results: DataFrame from test_thresholds function
        metric: Metric to optimize ('f1_score', 'precision', 'recall', 'accuracy')
    
    Returns:
        Optimal threshold value and corresponding metrics
    """
    optimal_idx = threshold_results[metric].idxmax()
    optimal_row = threshold_results.loc[optimal_idx]
    
    return optimal_row['threshold'], optimal_row

def print_threshold_analysis(threshold_results, y_true, y_proba):
    """Print comprehensive threshold analysis"""
    print("\n" + "="*80)
    print("                    THRESHOLD ANALYSIS RESULTS")
    print("="*80)
    
    # Find optimal thresholds for different metrics
    metrics = ['f1_score', 'precision', 'recall', 'accuracy']
    optimal_thresholds = {}
    
    for metric in metrics:
        threshold, row = find_optimal_threshold(threshold_results, metric)
        optimal_thresholds[metric] = (threshold, row)
    
    # Print optimal thresholds summary
    print("\nOPTIMAL THRESHOLDS BY METRIC:")
    print("-" * 50)
    for metric, (threshold, row) in optimal_thresholds.items():
        print(f"{metric.upper():12}: {threshold:.3f} "
              f"(P={row['precision']:.3f}, R={row['recall']:.3f}, "
              f"F1={row['f1_score']:.3f}, Acc={row['accuracy']:.3f})")
    
    # Print detailed results for key thresholds
    print(f"\nDETAILED THRESHOLD ANALYSIS:")
    print("-" * 50)
    key_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    
    print(f"{'Thresh':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6} "
          f"{'Spec':>6} {'TP':>4} {'FP':>4} {'TN':>4} {'FN':>4} {'Pred+':>5}")
    print("-" * 80)
    
    for threshold in key_thresholds:
        row = threshold_results[threshold_results['threshold'].round(3) == threshold]
        if not row.empty:
            row = row.iloc[0]
            print(f"{row['threshold']:6.2f} {row['precision']:6.3f} {row['recall']:6.3f} "
                  f"{row['f1_score']:6.3f} {row['accuracy']:6.3f} {row['specificity']:6.3f} "
                  f"{row['tp']:4.0f} {row['fp']:4.0f} {row['tn']:4.0f} {row['fn']:4.0f} "
                  f"{row['predicted_positive']:5.0f}")
    
    # Calculate baseline metrics
    baseline_precision = np.mean(y_true)  # Random precision would be proportion of positives
    print(f"\nBASELINE METRICS:")
    print("-" * 50)
    print(f"Random Baseline Precision: {baseline_precision:.3f}")
    print(f"Average Precision Score: {average_precision_score(y_true, y_proba):.3f}")
    print(f"AUC-ROC Score: {roc_auc_score(y_true, y_proba):.3f}")
    
    # Recommendations
    print(f"\nRECOMMENDations:")
    print("-" * 50)
    f1_threshold = optimal_thresholds['f1_score'][0]
    precision_threshold = optimal_thresholds['precision'][0]
    recall_threshold = optimal_thresholds['recall'][0]
    
    print(f"• For balanced performance: Use threshold {f1_threshold:.3f} (optimizes F1-score)")
    print(f"• For high precision (fewer false positives): Use threshold {precision_threshold:.3f}")
    print(f"• For high recall (catch more winners): Use threshold {recall_threshold:.3f}")
    
    return optimal_thresholds

# ============================================================
#                    MAIN PIPELINE
# ============================================================

def main():
    """Main training pipeline with threshold testing"""
    
    # Load data
    print("Loading data...")
    df = pd.read_csv("merged_horse_racing_data.csv")
    
    # Process features
    df = process_features(df)
    
    # Prepare data for TabNet
    X, y, feature_names, attention_mask = prepare_data_for_tabnet(df)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
    )
    
    print(f"Training set size: {X_train.shape}")
    print(f"Validation set size: {X_val.shape}")
    print(f"Test set size: {X_test.shape}")
    print(f"Class distribution - Train: {np.mean(y_train):.3f}, Val: {np.mean(y_val):.3f}, Test: {np.mean(y_test):.3f}")
    
    # Train TabNet
    print("\nTraining TabNet...")
    model = train_tabnet(X_train, y_train, X_val, y_val, attention_mask)
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_preds = model.predict_proba(X_test)[:, 1]
    test_accuracy = np.mean((test_preds > 0.5) == y_test)
    test_auc = roc_auc_score(y_test, test_preds)
    
    print(f"\nBasic Test Results:")
    print(f"Test Accuracy (0.5 threshold): {test_accuracy:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    
    # Threshold Testing
    print("\nPerforming threshold analysis...")
    threshold_results = test_thresholds(y_test, test_preds)
    
    # Print comprehensive threshold analysis
    optimal_thresholds = print_threshold_analysis(threshold_results, y_test, test_preds)
    
    # Plot precision-recall curve
    print("\nGenerating precision-recall curve...")
    avg_precision = plot_precision_recall_curve(y_test, test_preds)
    
    # Feature importance
    print("\nFeature Importance (Top 20):")
    print("-" * 50)
    importances = model.feature_importances_
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(importance_df.head(20))
    
    # Save results
    model.save_model('tabnet_horse_racing_model')
    importance_df.to_csv('feature_importance.csv', index=False)
    threshold_results.to_csv('threshold_analysis.csv', index=False)
    
    print(f"\nFiles saved:")
    print(f"• Model: tabnet_horse_racing_model.zip")
    print(f"• Feature importance: feature_importance.csv")
    print(f"• Threshold analysis: threshold_analysis.csv")
    print(f"• Precision-recall curve: precision_recall_curve.png")
    
    return model, feature_names, importance_df, threshold_results, optimal_thresholds

if __name__ == "__main__":
    model, features, importance, threshold_results, optimal_thresholds = main()

In [ ]:
def form_calc(df):
    df.sort('race_date')
    df.groupby('horse_name')['position_clean']


df == df.apply(form_calc)